<a href="https://colab.research.google.com/github/GowcikS/GenAI-exp/blob/main/Exp10(Gen_AI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INSTALL
# ============================================================

!pip install -q -U datasets transformers accelerate scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.2 MB/s eta 0:00:00


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# ============================================================
# DISTILBERT FINE-TUNING ON IMDB USING T4 GPU
# ============================================================

import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score


# ============================================================
# 1. VERIFY GPU
# ============================================================

print("=" * 60)
print("GPU CHECK")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU is not available!")


# ============================================================
# 2. LOAD IMDB DATASET
# ============================================================

print("\n" + "=" * 60)
print("LOADING IMDB DATASET")
print("=" * 60)

dataset = load_dataset("stanfordnlp/imdb")

# Use a smaller subset for faster training
small_train = (
    dataset["train"]
    .shuffle(seed=42)
    .select(range(2000))
)

small_test = (
    dataset["test"]
    .shuffle(seed=42)
    .select(range(500))
)

print("Training samples:", len(small_train))
print("Testing samples :", len(small_test))


# ============================================================
# 3. LOAD TOKENIZER
# ============================================================

print("\nLoading DistilBERT tokenizer...")

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)


# ============================================================
# 4. TOKENIZE DATASET
# ============================================================

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


print("Tokenizing training dataset...")

train_ds = small_train.map(
    tokenize,
    batched=True
)

print("Tokenizing testing dataset...")

test_ds = small_test.map(
    tokenize,
    batched=True
)


# ============================================================
# 5. LOAD DISTILBERT MODEL
# ============================================================

print("\nLoading DistilBERT classification model...")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


# ============================================================
# 6. CHECK MODEL DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Model device:", next(model.parameters()).device)


# ============================================================
# 7. TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=2,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=50,

    report_to="none",

    fp16=True
)


# ============================================================
# 8. ACCURACY METRIC
# ============================================================

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(
        predictions,
        axis=1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy
    }


# ============================================================
# 9. CREATE TRAINER
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_ds,
    eval_dataset=test_ds,

    compute_metrics=compute_metrics
)


# ============================================================
# 10. TRAIN
# ============================================================

print("\n" + "=" * 60)
print("STARTING GPU TRAINING")
print("=" * 60)

trainer.train()


# ============================================================
# 11. EVALUATE
# ============================================================

print("\n" + "=" * 60)
print("EVALUATION")
print("=" * 60)

metrics = trainer.evaluate()

print("\nEvaluation metrics:")

for key, value in metrics.items():
    print(f"{key}: {value}")


# ============================================================
# 12. SAVE FINE-TUNED MODEL
# ============================================================

output_path = "./fine_tuned_distilbert_imdb"

model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)

print("\n" + "=" * 60)
print("MODEL SAVED")
print("=" * 60)

print("Location:", output_path)

GPU CHECK
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

LOADING IMDB DATASET


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...


EVALUATION


Training Loss,Validation Loss,Epoch,Accuracy
0.222961,0.498368,2,0.806000



Evaluation metrics:
eval_loss: 0.4983678162097931
eval_accuracy: 0.806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MODEL SAVED
Location: ./fine_tuned_distilbert_imdb
